In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
import openpyxl
print(openpyxl.__version__)

inport Employees Table

In [0]:
from openpyxl import load_workbook

file_path= "/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)

sheet=workbook["employees"]

data=list(sheet.values)

headers=data[0]

rows=data[1:]

print(headers)
print("Number of rows:",len(rows))

employees=spark.createDataFrame(rows,schema=headers)
display(employees)


import Employees_service table

In [0]:
from openpyxl import load_workbook

file_path="/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)
sheet=workbook["employee_service"]
data=list(sheet.values)

headers=data[0]

rows=data[1:]
print(headers)
print("Number of rows",len(rows))

employee_service=spark.createDataFrame(rows,schema=headers)
display(employee_service)

In [0]:
orders=spark.read.csv("/Volumes/main/training/csv/orders.csv",header=True,inferSchema=True)
display(orders)

Question 1. Input Tables / Columns: employee_service (emp_id, hire_date, resignation_date)
Task Requirements:
•	Calculate total calendar days between hire date and resignation date.
•	For active employees whose resignation date is missing, measure tenure dynamically up to today's date.
•	Display employee identifier, hire date, resignation date, and computed service days.
•	Sort workforce records from longest tenure to shortest tenure.
Expected Output Specification: DataFrame with columns [emp_id, hire_date, resignation_date, total_service_days], ordered by service days descending.
Guiding Hints & Formulas: Formula / Concept: Service Days = Coalesce(Resignation Date, Today) - Hire Date. Investigate PySpark date difference and null fallback functions.


In [0]:
from pyspark.sql.functions import col,coalesce,current_date,datediff

result=( 
    employee_service
    .withColumn(
        "total_service_days",
        datediff(
            coalesce(col("resignation_date"),current_date()),
            col("hire_date")
        )
).select(
    "emp_id",
    "hire_date",
    "resignation_date",
    "total_service_days"
    
    ).orderBy(col("total_service_days").desc())
)

display(result)

Question 2. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Decompose each employee's hire date into its constituent calendar components: calendar year, month number, day of month, day of week, day of year, and fiscal quarter.
•	Ensure day of week indicates position within the standard week.
•	Display employee identifier and hire date alongside all extracted calendar fields.
Expected Output Specification: DataFrame with 8 columns including employee identifier, hire date, and all individual calendar components.
Guiding Hints & Formulas: Formula / Concept: Explore PySpark temporal extraction functions for year, month, day, weekday, day-of-year, and quarter.


In [0]:
from pyspark.sql.functions import year,day,month,quarter,dayofyear,dayofweek,dayofmonth


result=employees.select(
    "employee_id",
    "hire_date",
    year(col("hire_date")).alias("Year"),
    month(col("hire_date")).alias("Month"),
    dayofmonth(col("hire_date")).alias("Day"),
    dayofweek(col("hire_date")).alias("Day Week"),
    dayofyear(col("hire_date")).alias("Day year"),
    quarter(col("hire_date")).alias("Quarter")
)

display(result)

Question 3. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Convert hire date into standard US numerical format (Month/Day/Year).
•	Convert hire date into formal corporate text format (Abbreviated Month Name Day, Full Year).
•	Convert hire date into an executive long-form presentation string (Full Day Name, Full Month Name Day, Full Year).
Expected Output Specification: DataFrame with columns [employee_id, hire_date, us_format, formal_format, full_text_format].
Guiding Hints & Formulas: Formula / Concept: Explore date formatting pattern strings in PySpark (e.g. 'MM/dd/yyyy', 'MMM dd, yyyy', 'EEEE, MMMM dd, yyyy').


In [0]:
from pyspark.sql.functions import date_format,col

result=employees.select(
    "employee_id",
    "hire_date",
    date_format("hire_date","MM/dd/yyyy").alias("us_format"),
    date_format("hire_date","MMM dd,yyyy").alias("formal_format"),
    date_format("hire_date","EEEE, MMMM dd, yyyy").alias("full_text_format")
)

display(result)

Question 4.Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Project the 90-calendar-day initial probation end date from the employee's hire date.
•	Project the 180-calendar-day mid-year performance review milestone.
•	Calculate an early-warning deadline exactly 14 calendar days prior to the probation end date for HR audit.
Expected Output Specification: DataFrame with columns [employee_id, hire_date, probation_end_date, mid_year_review_date, probation_warning_deadline].
Guiding Hints & Formulas: Formula / Concept: Milestone Date = Base Date + Number of Days. Warning Date = Milestone Date - Offset Days.


In [0]:
from pyspark.sql.functions import col,date_add

result=employees.withColumn(
    "probation_end_date",
    date_add("hire_date",90)
).withColumn(
    "mid_year_review_date",
    date_add("hire_date",180) 
).withColumn(
    "probation_warning_deadline",
    date_add("probation_end_date",-14)
).select(
    "employee_id",
    "hire_date",
    "probation_end_date",
    "mid_year_review_date",
    "probation_warning_deadline"
)

display(result)



Question 5. **Input Tables / Columns:** employees (employee_id, hire_date)

**Task Requirements:**

·       Determine the calendar start date (first day of the month) corresponding to each employee's hire date.

·       Determine the calendar end date (last day of the month) for that same month.

·       Calculate the total calendar days contained in that specific hiring month.

·       Calculate how many calendar days remained in the hiring month from the employee's start date onwards.

**Expected Output Specification:** DataFrame with columns [employee_id, hire_date, month_start_date, month_end_date, total_month_days, remaining_hire_days].

**Guiding Hints & Formulas:** *Formula / Concept: Total Month Days = (Last Day of Month - First Day of Month) + 1. Remaining Days = (Last Day - Hire Date) + 1.*    

In [0]:
from pyspark.sql.functions import col,trunc,last_day,datediff


result=employees.withColumn(
    "month_start_date",
    trunc("hire_date","month")
).withColumn(
    "month_end_date",
    last_day("hire_date")
).select(
    "employee_id",
    "hire_date",
    "month_start_date",
    "month_end_date",
    (datediff(
            col("month_end_date"),
            col("month_start_date")
        )+1
    ).alias("total_month_days"),
    (datediff(
            col("month_end_date"),
            col("hire_date")
        )+1
    ).alias("remaining_hire_days"),
)

display(result)